In [191]:
# Finally, We are on #5 MEMORY! Let's goooo

# If you don't add 'memory' to chatbot, chatbot can't remember anything..
# even following questions, chatbot dont understand if dont have memory => stateless

from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(return_messages=True)

memory.save_context({"input": "Hi!"},{"output": "How are you?"})

memory.load_memory_variables({})

# ConversationBufferMemory save all conversation. it's inefficient. 

{'history': [HumanMessage(content='Hi!'), AIMessage(content='How are you?')]}

In [192]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    return_messages=True,
    k=4
)

def add_message(input, output):
    memory.save_context({"input":input}, {"output":output})

add_message(1, 1)
add_message(2, 2)
add_message(3, 3)
add_message(4, 4)

memory.load_memory_variables({})


{'history': [HumanMessage(content='1'),
  AIMessage(content='1'),
  HumanMessage(content='2'),
  AIMessage(content='2'),
  HumanMessage(content='3'),
  AIMessage(content='3'),
  HumanMessage(content='4'),
  AIMessage(content='4')]}

In [193]:
add_message(5,5)
memory.load_memory_variables({})
# ConversationBufferWindowMemory fix conversation size always. it's good to make efficient, but chatbot cant remember old messages..

{'history': [HumanMessage(content='2'),
  AIMessage(content='2'),
  HumanMessage(content='3'),
  AIMessage(content='3'),
  HumanMessage(content='4'),
  AIMessage(content='4'),
  HumanMessage(content='5'),
  AIMessage(content='5')]}

In [194]:
# now, we use llm with memory. it costs.
# ConversationSummaryMemory: summary of the conversation automatically
from langchain.memory import ConversationSummaryMemory
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryMemory(llm=llm)

def get_history():
    return memory.load_memory_variables({})

add_message("Hi, My name is Sofia Kim. i live in South Korea", "Wow that is so cool")

In [195]:
add_message("Korea is pretty", "I wish to go there")

get_history()

{'history': 'The human introduces herself as Sofia Kim from South Korea. The AI responds with enthusiasm about her location and name, expressing a desire to visit Korea because it is pretty.'}

In [196]:
# ConversationSummaryBufferMemory = window buffer memory + buffer window memory
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=50,
    return_messages=True,
)
# max_token_limit is max size before summery (original conversation max)

add_message("Hi, My name is Sofia Kim. i live in Japan", "Wow that is so cool")
get_history()

{'history': [HumanMessage(content='Hi, My name is Sofia Kim. i live in Japan'),
  AIMessage(content='Wow that is so cool')]}

In [197]:
add_message("Japan weather is warm", "I wish I could go!")
get_history()

{'history': [HumanMessage(content='Hi, My name is Sofia Kim. i live in Japan'),
  AIMessage(content='Wow that is so cool'),
  HumanMessage(content='Japan weather is warm'),
  AIMessage(content='I wish I could go!')]}

In [198]:
add_message("How far Japan to Korea?", "near")
get_history()

{'history': [SystemMessage(content='The human introduces herself as Sofia Kim and mentions she lives in Japan.'),
  AIMessage(content='Wow that is so cool'),
  HumanMessage(content='Japan weather is warm'),
  AIMessage(content='I wish I could go!'),
  HumanMessage(content='How far Japan to Korea?'),
  AIMessage(content='near')]}

In [199]:
add_message("Korea dish is so delicious", "Oh I like Korean Food!")
get_history()

{'history': [SystemMessage(content='The human introduces herself as Sofia Kim and mentions she lives in Japan. The AI expresses admiration for this and Sofia mentions the warm weather in Japan, prompting the AI to express a desire to visit.'),
  HumanMessage(content='How far Japan to Korea?'),
  AIMessage(content='near'),
  HumanMessage(content='Korea dish is so delicious'),
  AIMessage(content='Oh I like Korean Food!')]}

In [200]:
# ConversationKGMemory : it choose/extract 'entity' from your conversation
from langchain.memory import ConversationKGMemory
from langchain.chat_models import ChatOpenAI

memory = ConversationKGMemory(
    llm=llm,
    return_messages=True,
)

add_message("Hi, My name is Sofia Kim. i live in Japan", "Wow that is so cool")

In [201]:
memory.load_memory_variables({"input": "who is Sofia Kim"})

{'history': [SystemMessage(content='On Sofia Kim: Sofia Kim lives in Japan.')]}

In [202]:
add_message("Sofia likes Kimchi", "Wow that is so cool")
memory.load_memory_variables({"input": "What does she like"})

{'history': [SystemMessage(content='On Sofia Kim: Sofia Kim lives in Japan.')]}

In [203]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history"
)

# we need 'space' for memory. that is 'chat_history'(variable)
template = """
    You are a helpful AI talking to a human.

    {chat_history}
    Human:{question}
    You:
"""

chain = LLMChain(
    llm=llm,
    memory=memory,
    verbose=True,
    prompt=PromptTemplate.from_template(template)
)

chain.predict(question="My name is Sofia Kim")
chain.predict(question="I live in Seoul. it's beautiful city")




> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    
    Human:My name is Sofia Kim
    You:


> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    Human: My name is Sofia Kim
AI: Hello Sofia Kim! How can I assist you today?
    Human:I live in Seoul. it's beautiful city
    You:


> Finished chain.


"That's wonderful to hear! Seoul is indeed a beautiful city with a rich history and vibrant culture. Is there anything specific you would like to know or discuss about Seoul?"

In [204]:
chain.predict(question="What is my name?")



> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    Human: My name is Sofia Kim
AI: Hello Sofia Kim! How can I assist you today?
Human: I live in Seoul. it's beautiful city
AI: That's wonderful to hear! Seoul is indeed a beautiful city with a rich history and vibrant culture. Is there anything specific you would like to know or discuss about Seoul?
    Human:What is my name?
    You:


> Finished chain.


'Your name is Sofia Kim.'

In [205]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
# memory class can out two types. 
# One type is string, and One type is message
memory.load_memory_variables({}) # you can see print just 'text'(string)

# If you change output type to message, follow below parameter : return_message
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history",
    return_messages=True
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a helpful AI talking to a human."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

chain = LLMChain(
    llm=llm,
    memory=memory,
    verbose=True,
    prompt=prompt
)


chain.predict(question="My name is Sofia Kim") # You can see print type 'message'
chain.predict(question="I live in Seoul. it's beautiful city")
chain.predict(question="What is my name?")



> Entering new LLMChain chain...
Prompt after formatting:
System: you are a helpful AI talking to a human.
Human: My name is Sofia Kim

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
System: you are a helpful AI talking to a human.
Human: My name is Sofia Kim
AI: Nice to meet you, Sofia Kim! How can I assist you today?
Human: I live in Seoul. it's beautiful city

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
System: you are a helpful AI talking to a human.
Human: My name is Sofia Kim
AI: Nice to meet you, Sofia Kim! How can I assist you today?
Human: I live in Seoul. it's beautiful city
AI: Seoul is indeed a beautiful city with a rich history and vibrant culture. Is there anything specific you enjoy about living in Seoul?
Human: What is my name?

> Finished chain.


'Your name is Sofia Kim.'

In [206]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history",
    return_messages=True
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a helpful AI talking to a human."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

def load_memory(_): 
    # if you dont put any parameter, it will cause error. 
    # because it should given 'question'(=input). every memory class given input, and through out output. remember! 
    return memory.load_memory_variables({})["chat_history"]

chain = RunnablePassthrough.assign(chat_history=load_memory) | prompt | llm # you can use several functions in RunnablePassthrough class

def invoke_chain(question):
    result = chain.invoke({
        "question": question
    })
    memory.save_context({"input": question},{"output": result.content})
    print(result)

invoke_chain("My name is Sofia")
invoke_chain("What is my name?")

# So far, we learned 3 ways to add 'memory' in 'prompt'
# 1. using 'LLM chain'
# 2. using 'Chat prompt template'
# 3. using 'manual memory management' (it's not auto, but this way could be better and efficient)

content='Nice to meet you, Sofia! How can I assist you today?'
content='Your name is Sofia.'


In [207]:
# RAG(Retrieval Augmented Generation)
# your own data + general data = prompt => model

# Retrival : Langchain module
# Retriving data course : source data => load data => split data (transform) / => embed (talk about later) => store
# loader : extract data from source and give them to langchain

from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import TextLoader, PyPDFLoader, UnstructuredFileLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter

splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

# UnstructuredFileLoader can load any type of file
loader = UnstructuredFileLoader("./files/chapter_one.txt")


# It's too big file, we need to split out. there is 2 ways
# 1.
# docs = loader.load()
# splitter.split_documents(docs)
# 2.
loader.load_and_split(text_splitter=splitter)
len(loader.load_and_split(text_splitter=splitter))


Created a chunk of size 963, which is longer than the specified 600
Created a chunk of size 774, which is longer than the specified 600
Created a chunk of size 954, which is longer than the specified 600
Created a chunk of size 922, which is longer than the specified 600
Created a chunk of size 1168, which is longer than the specified 600
Created a chunk of size 821, which is longer than the specified 600
Created a chunk of size 700, which is longer than the specified 600
Created a chunk of size 745, which is longer than the specified 600
Created a chunk of size 735, which is longer than the specified 600
Created a chunk of size 1110, which is longer than the specified 600
Created a chunk of size 991, which is longer than the specified 600
Created a chunk of size 990, which is longer than the specified 600
Created a chunk of size 1182, which is longer than the specified 600
Created a chunk of size 1491, which is longer than the specified 600
Created a chunk of size 1401, which is longe

45

In [208]:
# same way to count text
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator='\n',
    chunk_size=600,
    chunk_overlap=100,
)


loader = UnstructuredFileLoader("./files/chapter_one.docx")

In [209]:
# embeding words
# only 3 vectors
'''
        Masculinity | Femininity | Royalty

king | 0.9 | 0.1 | 1.0
queen | 0.1 | 0.9 | 1.0
man | 0.9 | 0.1 | 0.0
woman | 0.1 | 0.9 | 0.0 

king - man = royal | 0.0 | 0.0 | 1.0
royal + woman = queen | 0.1 | 0.9 | 1.0
knight | 1.0 | 0.0 | 0.8

'''

# checkout this word similar site : https://turbomaze.github.io/word2vecjson/?utm_source=chatgpt.com 

'\n        Masculinity | Femininity | Royalty\n\nking | 0.9 | 0.1 | 1.0\nqueen | 0.1 | 0.9 | 1.0\nman | 0.9 | 0.1 | 0.0\nwoman | 0.1 | 0.9 | 0.0 \n\nking - man = royal | 0.0 | 0.0 | 1.0\nroyal + woman = queen | 0.1 | 0.9 | 1.0\nknight | 1.0 | 0.0 | 0.8\n\n'

In [210]:
# take a look about embedding model 
from langchain.embeddings import OpenAIEmbeddings

embedder = OpenAIEmbeddings()
# you can embed only documents, but also 'query'
vector = embedder.embed_documents([
    "hi",
    "how are you",
    "longer",
    "sentences",
    "as you can..."
])

In [211]:
# save embedding (caching) => to save money 
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma, FAISS
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore

llm = ChatOpenAI()

cache_dir = LocalFileStore("./.cache/")


splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

# UnstructuredFileLoader can load any type of file
loader = UnstructuredFileLoader("./files/chapter_one.txt")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

vectorstore = FAISS.from_documents(docs, cached_embeddings)



Created a chunk of size 963, which is longer than the specified 600
Created a chunk of size 774, which is longer than the specified 600
Created a chunk of size 954, which is longer than the specified 600
Created a chunk of size 922, which is longer than the specified 600
Created a chunk of size 1168, which is longer than the specified 600
Created a chunk of size 821, which is longer than the specified 600
Created a chunk of size 700, which is longer than the specified 600
Created a chunk of size 745, which is longer than the specified 600
Created a chunk of size 735, which is longer than the specified 600
Created a chunk of size 1110, which is longer than the specified 600
Created a chunk of size 991, which is longer than the specified 600
Created a chunk of size 990, which is longer than the specified 600
Created a chunk of size 1182, which is longer than the specified 600
Created a chunk of size 1491, which is longer than the specified 600
Created a chunk of size 1401, which is longe

In [212]:
results = vectorstore.similarity_search("where does winston live?")
print(results)

[Document(page_content="Winston turned round abruptly. He had set his features into the expression of quiet optimism which it was advisable to wear when facing the telescreen. He crossed the room into the tiny kitchen. By leaving the Ministry at this time of day he had sacrificed his lunch in the canteen, and he was aware that there was no food in the kitchen except a hunk of dark-coloured bread which had got to be saved for tomorrow's breakfast. He took down from the shelf a bottle of colourless liquid with a plain white label marked VICTORY GIN. It gave off a sickly, oily smell, as of Chinese ricespirit. Winston poured out nearly a teacupful, nerved himself for a shock, and gulped it down like a dose of medicine.", metadata={'source': './files/chapter_one.txt'}), Document(page_content='Winston kept his back turned to the telescreen. It was safer, though, as he well knew, even a back can be revealing. A kilometre away the Ministry of Truth, his place of work, towered vast and white ab

In [213]:
# off-the-shelf chain (it's not legacy?)
from langchain.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",
    retriever=vectorstore.as_retriever(),
)


chain.run("Where does Winston live?")
# chain.run("Describe Victory Mansions")

# ==> document GPT

"The additional context provides a clearer picture of Winston's living situation. In his apartment at Victory Mansions, Winston strategically positions himself in an alcove to avoid being directly seen by the telescreen in his living room. This detail emphasizes the constant surveillance and lack of privacy in Winston's living environment."

In [217]:
# Make your own stuff chain and map reduce chain using LCEL
# first, we gonna make our own stuff (using LCEL) => expose on the code!
# Stuff LCEL Chain
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(
    temperature=0.1
)

retriver = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer"
    "just say you don't know, don't make it up:\n\n{context}"),
    ("human", "{question}")
])

chain = {"context": retriver, "question": RunnablePassthrough()} | prompt | llm

chain.invoke("Describe Victory Mansions")

AIMessage(content="Victory Mansions is a building with glass doors that let in gritty dust. It is a place where Winston Smith lives, and it has a hallway that smells of boiled cabbage and old rag mats. The building is seven flights up, with a non-functional lift due to the economy drive. There is a large colored poster of a man's face on the wall, and the building overlooks other similar structures in London.")

In [ ]:

# implement MapReduce chain type using LCEL (not using RetriverQA) => how does Mapredue it work?
from langchain.schema.runnable import RunnableLambda


retriver = vectorstore.as_retriever()
'''
list of docs

for doc in list of docs | prompt | lllm 

for response in list of llms response | put them all together

final-docs | prompt | llm 
'''

map_doc_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Use the following portion of a long document to see if any of the text is relevant to answer the question.
        Return any relevant text verbatim.
        ------
        {context}
        """,
    ),
    ("human", "{question}"),
])

map_doc_chain = map_doc_prompt | llm


def map_docs(inputs):
    documents = inputs['documents']
    question = inputs['question']
    # results = []
    # for document in documents:
    #     result = map_doc_chain.invoke({
    #         "context":document.page_content,
    #         "question": question
    #     }).content
    #     results.append(result)
    #     print(result)
    # results = "\n\n".join(results)
    # return results
    return "\n\n".join(map_doc_chain.invoke({
        "context": doc.page_content,
        "question":question
    }).content for doc in documents)

# map_chain = "looooong text from original document(each answers for question)"
map_chain = { "documents": retriver, "question": RunnablePassthrough() } | RunnableLambda(map_docs)

final_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """
     Given the following extracted parts of a long document and a question,
     create a final answer.
     If you don't know the answer, just say that you don't know. Don't try to make up an answer.
     ------
    {context}
     """),
     ("human", "{question}")
])

chain = {"context": map_chain, "question": RunnablePassthrough()} | final_prompt | llm

chain.invoke("Where does Winston go to work?")

Winston Smith works at the Ministry of Truth.
Winston goes to work in the Records Department.
Winston works at the Ministry, as mentioned in the text: "By leaving the Ministry at this time of day he had sacrificed his lunch in the canteen."
Winston goes to work at the Ministry of Truth, which is described as towering vast and white above the grimy landscape. It is located a kilometre away from where he stands with his back turned to the telescreen. The Ministry of Truth is his place of work in London, the chief city of Airstrip One, which is the third most populous province of Oceania.


AIMessage(content='Winston goes to work at the Ministry of Truth.')